In [0]:
from pyspark.sql import functions as F
from datetime import datetime

start_time = datetime.now()

In [0]:
def audit_log(run_id, schema_name, table_name, start_time, status, row_count, error):
    end_time = datetime.now()
    log = {
        'run_id': run_id,
        'schema_name': schema_name,
        'table_name': table_name,
        'start_time': str(start_time),
        'end_time': str(end_time),
        'duration': (end_time - start_time).total_seconds(),
        'status': status,
        'rows_written': int(row_count or 0),
        'error_message': error
    }
    schema = ("run_id string, schema_name string, table_name string, "
              "start_time string, end_time string, duration double, "
              "status string, rows_written bigint, error_message string")

    spark.createDataFrame([log], schema) \
         .write.mode('append').saveAsTable('retail.ops.audit_logs')

def update_watermark(schema, table, latest_watermark):
    if latest_watermark != '' or latest_watermark is None:   # no new rows → leave the watermark unchanged
      try:
        spark.sql(f"""
          UPDATE retail.ops.watermark
          SET last_watermark = '{latest_watermark}'
          WHERE schema_name = '{schema}' AND table_name = '{table}'
        """)
      except Exception as e:
        return None

### Setup Task Params

In [0]:
# dbutils.widgets.text('catalog', 'retail', 'Catalog')
# dbutils.widgets.text('schema', 'bronze', 'Schema')
# dbutils.widgets.text('table', 'orders', 'Table')
# dbutils.widgets.text('run_id', '', 'Run ID')
# dbutils.widgets.text('source_path', '/Volumes/retail/bronze/landing/transaction/', 'Source Path')
# dbutils.widgets.text('watermark_column', 'txn_ts', 'Watermark Column')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
table = dbutils.widgets.get('table')
run_id = dbutils.widgets.get('run_id')
source_path = dbutils.widgets.get('source_path')
watermark_column = dbutils.widgets.get('watermark_column')

### Get the Latest Watermark for incremental Load

In [0]:
latest_watermark = spark.sql(
    f"SELECT MAX(last_watermark) FROM retail.ops.watermark WHERE schema_name = 'bronze' AND table_name = '{table}'"
)
latest_watermark = latest_watermark.first()[0]

## Read the Table

In [0]:

## Read files from landing if any new
df = (
    spark.read.format('csv')
    .options(header=True)
    .load(f'{source_path}')
    .withColumn('_ingested_at', F.current_timestamp())
    .withColumn('_file_source', F.col('_metadata.file_path'))
    .withColumn(f'{watermark_column}', F.col('_metadata.file_modification_time'))
    .withColumn('run_id', F.lit(run_id))
)

df = df.filter(F.col(f'{watermark_column}')> latest_watermark)

In [0]:
try:
    wm_row = spark.sql(f"""
        SELECT last_watermark FROM retail.ops.watermark
        WHERE schema_name = '{schema}' AND table_name = '{table}'
    """).first()
    
    if wm_row is None:
        raise ValueError(f"No watermark entry for {schema}.{table}. Check the for-each inputs or seed ops.watermark")

    # 2. The source path must exist
    try:
        dbutils.fs.ls(source_path)
    except Exception:
        raise FileNotFoundError(f"Source path not found: {source_path}")

    # 3. The watermark column must exist in the source data
    if watermark_column not in df.columns:
        raise ValueError(f"Column '{watermark_column}' not in source. Found: {df.columns}")

except Exception as e:
    audit_log(
        run_id=run_id, schema_name=schema, table_name=table,
        start_time=start_time, status='FAILED', row_count=0,
        error=f"VALIDATION: {str(e)[:980]}"
    )
    raise

### Write the table

In [0]:
status = None
error = None
rows_written = 0
try:
    stats = df.agg(
        F.max(f'{watermark_column}').alias('latest_watermark'),
        F.count('*').alias('row_count')
    ).first()

    row_count = stats['row_count']
    updated_watermark = stats['latest_watermark']

    if row_count > 0:
        (
            df
            .write
            .format('delta')
            .mode('append')
            .option('mergeSchema', 'true')
            .saveAsTable(f'{catalog}.{schema}.{table}')
        )
        rows_written = row_count

        if updated_watermark is not None:
            update_watermark(schema, table, stats['latest_watermark'])

    status = 'SUCCESS'

except Exception as e:
    status = 'FAILED'
    error = str(e)[:1000]
    raise

finally:
    audit_log(
        run_id=run_id,
        schema_name=schema,
        table_name=table,
        start_time=start_time,
        status=status,
        row_count=rows_written,
        error=error
    )